# Investigate HMO companies

First match every saved HMO page to Companies House. In a separate pass, expand the network through every matched company's officers. The name-search script controls whether ambiguous or missing results prompt for input.

In [1]:
from pathlib import Path
import re
import sqlite3

import pandas as pd

from search_data_by_company_name import search_data_by_company_name
from search_officers_by_company_number import DATABASE_PATH
from search_companies_by_officer_id import (
    search_companies_by_officer_id,
    store_data_from_search_by_officer_id,
)

HMO_PAGES = Path("hmo_pages")
HMO_DATABASE_PATH = Path("hmo_companies.db")

In [2]:
def query_all_hmo_pages():
    """Match every saved page and store its Companies House company."""
    page_files = sorted(HMO_PAGES.glob("*.html"))

    with sqlite3.connect(HMO_DATABASE_PATH) as hmo_conn:
        hmo_conn.execute(
            """
            CREATE TABLE IF NOT EXISTS hmo_companies (
                company_name TEXT NOT NULL,
                company_number TEXT NOT NULL,
                filename TEXT NOT NULL,
                PRIMARY KEY (filename, company_number)
            )
            """
        )
        hmo_conn.execute(
            """
            CREATE TABLE IF NOT EXISTS processed_pages (
                filename TEXT PRIMARY KEY,
                match_count INTEGER NOT NULL
            )
            """
        )
        hmo_conn.execute(
            """
            INSERT OR IGNORE INTO processed_pages (filename, match_count)
            SELECT filename, COUNT(*)
            FROM hmo_companies
            GROUP BY filename
            """
        )
        processed_files = {
            row[0]
            for row in hmo_conn.execute(
                """
                SELECT filename FROM processed_pages
                WHERE match_count > 0
                """
                
            )
        }

        for index, page_file in enumerate(page_files, 1):
            if page_file.name in processed_files:
                print(f"{index}/{len(page_files)}: skipping {page_file.stem}")
                continue
            page_name = re.sub(r"-\d+$", "", page_file.stem)
            search_name = page_name.replace("-", " ")
            print(f"{index}/{len(page_files)}: {search_name}")
            matches = search_data_by_company_name(search_name)

            hmo_conn.executemany(
                """
                INSERT INTO hmo_companies (
                    company_name, company_number, filename
                ) VALUES (?, ?, ?)
                ON CONFLICT(filename, company_number) DO UPDATE SET
                    company_name = excluded.company_name
                """,
                [
                    (
                        match["company_name"],
                        match["company_number"],
                        page_file.name,
                    )
                    for match in matches
                ],
            )
            hmo_conn.execute(
                """
                INSERT OR IGNORE INTO processed_pages (filename, match_count)
                VALUES (?, ?)
                """,
                (page_file.name, len(matches)),
            )
            hmo_conn.commit()

        return pd.read_sql_query(
            """
            SELECT company_name, company_number, filename
            FROM hmo_companies
            ORDER BY filename, company_number
            """,
            hmo_conn,
        )

## Pass 1: match the HMO pages

Run this pass first. It matches all 516 filenames and stores each selected Companies House company and its direct officers. Successfully processed filenames—including zero-match skips—are checkpointed and skipped on later runs.

In [3]:
hmo_companies = query_all_hmo_pages()
hmo_companies

1/516: skipping 1let
2/516: skipping 247-property
3/516: skipping 2let-agency
4/516: 2let2
5/516: skipping aberdein-considine
6/516: abode leeds
7/516: skipping abode-lettings
8/516: skipping abode-ltd
9/516: abode property management bristol
10/516: skipping abraham-estates
11/516: acara property management
12/516: access properties york
13/516: skipping adam-bennett-lettings
14/516: skipping admiral-estates
15/516: skipping airsat-real-estate
16/516: skipping albany-lettings
17/516: alexander greens
18/516: skipping alh-property-solutions
19/516: skipping am-living
20/516: skipping amber-court-lettings
21/516: ams housing group
22/516: anthony james co
23/516: skipping apex-estate-agency
24/516: skipping arch-living
25/516: archer bassett co
26/516: skipping arden-property-management
27/516: ariston property
28/516: arx arbor
29/516: skipping ashtons-estate-agents
30/516: skipping ask-property
31/516: aston woolf
32/516: skipping avenue-property-management
33/516: skipping avtar-prop

,company_name,company_number,filename
0,1LET LIMITED,SC317107,1let.html
1,247 PROPERTY LTD,13732862,247-property.html
2,247 PROPERTY LIMITED,15071229,247-property.html
3,2LET AGENCY LIMITED,07211683,2let-agency.html
4,2 LET 2 LTD,05309710,2let2.html
...,...,...,...
1009,ZEST LETTINGS LTD,15866430,zest-lettings.html
1010,ZEST PROPERTY MANAGEMENT LIMITED,06053537,zest-property-management.html
1011,ZONE LETTING LIMITED,SC243615,zone-letting-1.html
1012,ZONE LETTING LIMITED,SC243615,zone-letting.html


## Pass 2: expand every officer network

Run this only after pass 1 is complete. It includes every officer and every historical or current company appointment. Each successfully expanded officer is checkpointed and skipped on later runs.

In [2]:
def expand_all_hmo_officer_networks():
    """Store every company associated with every matched HMO officer."""
    with (
        sqlite3.connect(HMO_DATABASE_PATH) as hmo_conn,
        sqlite3.connect(DATABASE_PATH) as company_conn,
    ):
        hmo_conn.execute(
            """
            CREATE TABLE IF NOT EXISTS expanded_officers (
                officer_id TEXT PRIMARY KEY
            )
            """
        )
        expanded_officer_ids = {
            row[0]
            for row in hmo_conn.execute(
                "SELECT officer_id FROM expanded_officers"
            )
        }
        company_numbers = [
            row[0]
            for row in hmo_conn.execute(
                """
                SELECT DISTINCT company_number
                FROM hmo_companies
                ORDER BY company_number
                """
            )
        ]

        officer_ids = {
            row[0]
            for company_number in company_numbers
            for row in company_conn.execute(
                """
                SELECT DISTINCT officer_id
                FROM search_data_by_company_number
                WHERE company_number = ?
                """,
                (company_number,),
            )
        }

        pending_officer_ids = sorted(officer_ids - expanded_officer_ids)
        for index, officer_id in enumerate(pending_officer_ids, 1):
            print(f"{index}/{len(pending_officer_ids)}: {officer_id}")
            officer_payload = search_companies_by_officer_id(officer_id)
            store_data_from_search_by_officer_id(
                officer_payload, company_conn
            )
            company_conn.commit()
            hmo_conn.execute(
                "INSERT INTO expanded_officers (officer_id) VALUES (?)",
                (officer_id,),
            )
            hmo_conn.commit()

    return len(pending_officer_ids)

In [3]:
expanded_officer_count = expand_all_hmo_officer_networks()
expanded_officer_count

1/2381: GGIddMattEOnS2mAEuo4CLgihPQ
2/2381: GHEVkVvTNTQSPigEwx74pEaoamA
3/2381: GHcKPE1pY0yOBiyMS43eJdRCXtg
4/2381: GJeDLGzZbyGvDka6yZ5s08O9HWc
5/2381: GJuozVVdPmEHrAqLBcA0yRZ7U0c
6/2381: GNog3fFoQ5UVPu5kcRPKoa6dV7s
7/2381: GRWtF1SBl9CNkQVQAxFSltSLwAk
8/2381: GS4jXPwJ3dOe_GDV1IfwYhTlAy0
9/2381: GTxULAwmkjWZWa6X3JnXQCYTrWs
10/2381: GWckBw1zRJ1MyM3hDrPQsLZdHXY
11/2381: GY4yqT1J-3SOP9kcTZK8FHESS30
12/2381: Gbc9rHkPzeH7CPAYnsfUX3xHBlg
13/2381: Gc3hg4H44FhR-GSxaGFcVgpNjeU
14/2381: GcjKdKX-0P5NfsPMOVkWnYQbgAM
15/2381: GdEGdL_mMwnGWH-_RGaB5ToGUBM
16/2381: Gdg6qj8l_zGVdWtoUjEDN1hhiL0
17/2381: GhQW4tHHyfGqRG-320HTJLnf3D0
18/2381: GicDxeCXd_bnQ9hVfJeTIYZF358
19/2381: Gl0DrrBxvHSoNN1cW4LFYpo_uMc
20/2381: GlWM003SVkYvkG4OV6yJw1_qk_E
21/2381: GoR-BUEXVAsmTP69Da5sT3vyPmM
22/2381: GozDawMLlIFDQDu4g4LBk_AZE1c
23/2381: GpvxxG-cmximC_RV45fmSjklOQk
24/2381: Gt2RicKzpEB3mkfwDOCRP1jII34
25/2381: GtXQIrUTfjc7NbUk3VTdxciKDXI
26/2381: Gy0kdUqOhHE1k2UbqM6r4MAgZyE
27/2381: Gya3kLtxixlmfIbvhiUDusxSS_w
28/2381: G

KeyboardInterrupt: 